# SeamlessM4T v2 en Colab: verificación, métricas y demo

**Artículo:** Seamless Communication *et al.*, "Seamless: Multilingual Expressive and Streaming Speech Translation", arXiv:2312.05187 (2023).
**Pesos:** `facebook/seamless-m4t-v2-large` · **Repositorio del proyecto:** https://github.com/MiguelDiuza/seamless-traductor

Usar con GPU: *Entorno de ejecución → Cambiar tipo de entorno → GPU T4*.

In [ ]:
!git clone -q https://github.com/MiguelDiuza/seamless-traductor.git
%cd seamless-traductor
!pip -q install transformers==4.57.6 sentencepiece sacrebleu jiwer soundfile librosa streamlit plotly
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no')

## 1. Datos: FLEURS español con su traducción al inglés

In [ ]:
!python scripts/preparar_datos.py

## 2. Verificación: nuestra implementación paso a paso = implementación oficial
Compara memoria del encoder, pesos de atención, logits y texto (transcripción y traducción).

In [ ]:
!python scripts/verificar_equivalencia.py

## 3. Métricas: SeamlessM4T v2 vs. Whisper large-v3 (150 frases)

In [ ]:
!python scripts/evaluar.py --n 150

In [ ]:
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv('resultados/metricas.csv').round(3))
display(Image('resultados/metricas.png')); display(Image('resultados/bleu_por_duracion.png'))

Descargar los resultados para subirlos al repositorio:

In [ ]:
!zip -q -r resultados.zip resultados/metricas.csv resultados/predicciones.csv resultados/*.png
from google.colab import files; files.download('resultados.zip')

## 4. (Respaldo) Abrir la app Streamlit desde Colab
Si el computador de la sustentación no tiene memoria suficiente, la app puede correr aquí y abrirse con un enlace público temporal.

In [ ]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
!nohup streamlit run app.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &
!sleep 5 && nohup ./cloudflared tunnel --url http://localhost:8501 > tunel.log 2>&1 &
!sleep 10 && grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' tunel.log | head -1